# Example 0.13.3 / 1: Exercise 15-4: A Transportation Problem as a Transshipment Network

Companion notebook for **0.13.3 Exercises on Special Structures** in the AMPL lab Documentation, section "Exercise 15-4: A Transportation Problem as a Transshipment Network". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Exercise 15-4: A Transportation Problem as a Transshipment Network

Exercise 15-4(a) asks for a data file that solves the transportation problem of the book's
Figure 3-1 with the general transshipment model of Figure 15-2a. In the transshipment model every
city has a supply and a demand, and the balance constraint at city $k$ says that supply plus
inflow equals demand plus outflow (Network Models (AMPL lab documentation, section 10.1)). A transportation problem is the
special case in which origins only supply, destinations only demand, and every link runs from an
origin to a destination. In the data and run fragments, `ampl` has already read the model.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set CITIES;                                     # all nodes
set LINKS within {CITIES, CITIES};              # arcs (from, to)
param supply {CITIES} >= 0, default 0;          # tons entering at a city
param demand {CITIES} >= 0, default 0;          # tons leaving at a city
check: sum {i in CITIES} supply[i] = sum {j in CITIES} demand[j];  # balanced network
param cost {LINKS} >= 0;                        # dollars per ton
param capacity {LINKS} >= 0, default Infinity;  # tons per link

var Ship {(i,j) in LINKS} >= 0, <= capacity[i,j];  # tons shipped on a link

minimize Total_Cost: sum {(i,j) in LINKS} cost[i,j] * Ship[i,j];  # dollars

# supply plus inflow equals demand plus outflow at every city
subject to Balance {k in CITIES}:
    supply[k] + sum {(i,k) in LINKS} Ship[i,k] = demand[k] + sum {(k,j) in LINKS} Ship[k,j];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import pandas as pd

supply = {"GARY": 1400, "CLEV": 2600, "PITT": 2900}
demand = {"FRA": 900, "DET": 1200, "LAN": 600, "WIN": 400, "STL": 1700, "FRE": 1100, "LAF": 1000}
cost = pd.DataFrame(
    [[39, 14, 11, 14, 16, 82, 8], [27, 9, 12, 9, 26, 95, 17], [24, 14, 17, 13, 28, 99, 20]],
    index=list(supply), columns=list(demand),
)
ampl.set["CITIES"] = list(supply) + list(demand)
ampl.set["LINKS"] = [(i, j) for i in supply for j in demand]
ampl.param["supply"] = supply
ampl.param["demand"] = demand
ampl.param["cost"] = cost.stack().to_dict()

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, ampl.obj["Total_Cost"].value())

HiGHS 1.14.0: optimal solution; objective 196200
10 simplex iterations
0 barrier iterations
solved 196200.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set CITIES := GARY CLEV PITT FRA DET LAN WIN STL FRE LAF;
set LINKS := (GARY,*) FRA DET LAN WIN STL FRE LAF
             (CLEV,*) FRA DET LAN WIN STL FRE LAF
             (PITT,*) FRA DET LAN WIN STL FRE LAF;
param supply := GARY 1400  CLEV 2600  PITT 2900;
param demand := FRA 900  DET 1200  LAN 600  WIN 400  STL 1700  FRE 1100  LAF 1000;
param cost:  FRA  DET  LAN  WIN  STL  FRE  LAF :=
GARY          39   14   11   14   16   82    8
CLEV          27    9   12    9   26   95   17
PITT          24   14   17   13   28   99   20;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Total_Cost;

HiGHS 1.14.0: optimal solution; objective 196200
10 simplex iterations
0 barrier iterations
Total_Cost = 196200

